<a href="https://colab.research.google.com/github/BerkKilincer-dev/SoftIto/blob/main/ML/dengesiz_veri_agac_modelleri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dengesiz Veri Metrikleri ve Ağaç Tabanlı Topluluk Modelleri

**Bölüm A – Dengesiz veri:** Sınıflardan biri çok nadirse (dolandırıcılık, hastalık teşhisi, arıza tespiti) *accuracy* neden yanıltır? Precision, Recall ve F1 skorunu sıfırdan hesaplayıp eşik ayarını yapacağız.

**Bölüm B – Ağaç modelleri:** Tek karar ağacı → **Bagging** (Random Forest) → **Boosting** (Gradient Boosting, XGBoost, LightGBM). Gradient boosting'i önce elle yazıp sonra kütüphanelerle karşılaştıracağız.

> Colab'da `xgboost` ve `lightgbm` hazır kurulu gelir. Lokal ortamda gerekirse: `pip install xgboost lightgbm`

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             precision_recall_curve, average_precision_score, roc_auc_score)
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 100

---
# Bölüm A — Dengesiz Veri ve Doğru Metrikler

Bir kredi kartı dolandırıcılık senaryosu düşünelim: işlemlerin yalnızca **%3'ü** sahtekârlık.

In [ ]:
X, y = make_classification(n_samples=20_000, n_features=20, n_informative=6, n_redundant=4,
                           weights=[0.97, 0.03], flip_y=0.01, class_sep=0.9, random_state=42)

X_egt, X_tmp, y_egt, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_dgr, X_tst, y_dgr, y_tst = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

print("Sınıf dağılımı (tüm veri):", dict(zip(*np.unique(y, return_counts=True))))
print(f"Pozitif oranı: %{y.mean() * 100:.2f}")
print(f"Eğitim / doğrulama / test: {len(y_egt)} / {len(y_dgr)} / {len(y_tst)}")

### Accuracy tuzağı

Hiçbir şey öğrenmeyen, **her işleme "normal" diyen** bir model yazalım.

In [ ]:
aptal_model = DummyClassifier(strategy="most_frequent").fit(X_egt, y_egt)
tahmin = aptal_model.predict(X_tst)

print(f"Accuracy : {accuracy_score(y_tst, tahmin):.3f}   ← harika görünüyor!")
print(f"Recall   : {recall_score(y_tst, tahmin):.3f}   ← tek bir dolandırıcılığı bile yakalamadı")
print(f"F1       : {f1_score(y_tst, tahmin, zero_division=0):.3f}")

### Karmaşıklık matrisi ve metrikler

|  | Tahmin: Pozitif | Tahmin: Negatif |
|---|---|---|
| **Gerçek: Pozitif** | TP (doğru alarm) | FN (kaçırılan) |
| **Gerçek: Negatif** | FP (yanlış alarm) | TN |

| Metrik | Formül | Soru | Ne zaman önemli? |
|---|---|---|---|
| **Precision** | $\frac{TP}{TP+FP}$ | "Pozitif dediklerimin ne kadarı gerçekten pozitif?" | Yanlış alarm pahalıysa (spam filtresi, gereksiz operasyon) |
| **Recall** (duyarlılık) | $\frac{TP}{TP+FN}$ | "Gerçek pozitiflerin ne kadarını yakaladım?" | Kaçırmak pahalıysa (kanser, dolandırıcılık) |
| **F1** | $2\cdot\frac{P \cdot R}{P + R}$ | İkisinin harmonik ortalaması | Tek bir dengeli skor gerektiğinde |

Harmonik ortalama küçük olana yakındır: P = 1.0, R = 0.1 ise aritmetik ortalama 0.55 ama F1 = 0.18.

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_egt, y_egt)
tahmin = lr.predict(X_tst)

tn, fp, fn, tp = confusion_matrix(y_tst, tahmin).ravel()
precision_elle = tp / (tp + fp)
recall_elle = tp / (tp + fn)
f1_elle = 2 * precision_elle * recall_elle / (precision_elle + recall_elle)

print(f"TP = {tp}, FP = {fp}, FN = {fn}, TN = {tn}\n")
print(f"{'Metrik':10s} {'Elle':>8s} {'sklearn':>8s}")
print(f"{'Precision':10s} {precision_elle:8.3f} {precision_score(y_tst, tahmin):8.3f}")
print(f"{'Recall':10s} {recall_elle:8.3f} {recall_score(y_tst, tahmin):8.3f}")
print(f"{'F1':10s} {f1_elle:8.3f} {f1_score(y_tst, tahmin):8.3f}")
print(f"{'Accuracy':10s} {(tp + tn) / (tp + tn + fp + fn):8.3f} {accuracy_score(y_tst, tahmin):8.3f}")

### `class_weight="balanced"`

Kayıp fonksiyonunda azınlık sınıfının hatalarını **daha ağır** cezalandırır. Ağırlık: $w_c = \frac{n_{\text{örnek}}}{n_{\text{sınıf}} \cdot n_c}$

In [ ]:
lr_dengeli = make_pipeline(StandardScaler(),
                           LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_egt, y_egt)

fig, eksenler = plt.subplots(1, 2, figsize=(11, 4))
for ax, (isim, model) in zip(eksenler, [("LogisticRegression", lr), ("class_weight='balanced'", lr_dengeli)]):
    ConfusionMatrixDisplay.from_estimator(model, X_tst, y_tst, ax=ax, cmap="Blues",
                                          display_labels=["normal", "dolandırıcılık"], colorbar=False)
    ax.set_title(isim)
plt.tight_layout(); plt.show()

print(classification_report(y_tst, lr_dengeli.predict(X_tst), target_names=["normal", "dolandırıcılık"]))

### Eşik (threshold) ayarı ve Precision–Recall eğrisi

Sınıflandırıcılar aslında bir **olasılık** üretir; `predict` bu olasılığı varsayılan olarak **0.5** eşiğiyle keser. Eşiği düşürürsek recall artar, precision düşer.

> ⚠️ Eşiği **doğrulama** setinde seçiyoruz. Test setinde seçersek bu da bir veri sızıntısı olur!

In [ ]:
olasilik_dgr = lr.predict_proba(X_dgr)[:, 1]
precision, recall, esikler = precision_recall_curve(y_dgr, olasilik_dgr)
f1_degerleri = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
en_iyi_esik = esikler[np.argmax(f1_degerleri)]

fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.5))
eksenler[0].plot(esikler, precision[:-1], label="Precision")
eksenler[0].plot(esikler, recall[:-1], label="Recall")
eksenler[0].plot(esikler, f1_degerleri, label="F1", lw=2)
eksenler[0].axvline(0.5, color="gray", ls=":", label="varsayılan eşik 0.5")
eksenler[0].axvline(en_iyi_esik, color="red", ls="--", label=f"en iyi F1 eşiği = {en_iyi_esik:.2f}")
eksenler[0].set_xlabel("eşik"); eksenler[0].set_title("Eşiğe göre metrikler (doğrulama seti)")
eksenler[0].legend()

for isim, model in [("LogReg", lr), ("LogReg balanced", lr_dengeli)]:
    p, r, _ = precision_recall_curve(y_tst, model.predict_proba(X_tst)[:, 1])
    ap = average_precision_score(y_tst, model.predict_proba(X_tst)[:, 1])
    eksenler[1].plot(r, p, label=f"{isim} (AP = {ap:.3f})")
eksenler[1].axhline(y_tst.mean(), color="gray", ls=":", label=f"rastgele model (= pozitif oranı {y_tst.mean():.2f})")
eksenler[1].set_xlabel("Recall"); eksenler[1].set_ylabel("Precision")
eksenler[1].set_title("Precision–Recall eğrisi (test seti)")
eksenler[1].legend()
plt.tight_layout(); plt.show()

### Karşılaştırma tablosu

ROC-AUC dengesiz veride genellikle **iyimser** görünür çünkü çok sayıdaki TN, FP oranını küçük tutar. Nadir sınıf için **PR-AUC (Average Precision)** daha dürüst bir özet metriktir.

In [ ]:
def metrikler(isim, y_gercek, olasilik, esik=0.5):
    tahmin = (olasilik >= esik).astype(int)
    return {"Model": isim, "Eşik": esik,
            "Accuracy": accuracy_score(y_gercek, tahmin),
            "Precision": precision_score(y_gercek, tahmin, zero_division=0),
            "Recall": recall_score(y_gercek, tahmin),
            "F1": f1_score(y_gercek, tahmin),
            "ROC-AUC": roc_auc_score(y_gercek, olasilik),
            "PR-AUC": average_precision_score(y_gercek, olasilik)}

tablo = pd.DataFrame([
    metrikler("Dummy (hep 0)", y_tst, aptal_model.predict_proba(X_tst)[:, 1]),
    metrikler("LogReg", y_tst, lr.predict_proba(X_tst)[:, 1]),
    metrikler("LogReg balanced", y_tst, lr_dengeli.predict_proba(X_tst)[:, 1]),
    metrikler("LogReg + ayarlı eşik", y_tst, lr.predict_proba(X_tst)[:, 1], esik=round(float(en_iyi_esik), 3)),
]).round(3)
tablo.style.background_gradient(subset=["Precision", "Recall", "F1", "PR-AUC"], cmap="Greens")

---
# Bölüm B — Ağaç Modelleri: Bagging vs Boosting

| | **Bagging** (Random Forest) | **Boosting** (GBM, XGBoost, LightGBM) |
|---|---|---|
| Ağaçlar nasıl kurulur? | **Paralel** ve bağımsız | **Sıralı**, her ağaç öncekilerin hatasını düzeltir |
| Her ağacın verisi | Bootstrap örneklemi (iadeli rastgele seçim) + rastgele özellik alt kümesi | Tüm veri; hedef = önceki modelin **artıkları** (negatif gradyan) |
| Ağaç derinliği | Derin (düşük bias, yüksek varyans) | Sığ (2–8 seviye, "zayıf öğrenici") |
| Neyi azaltır? | **Varyansı** | **Bias'ı** (ve dikkatli ayarla varyansı) |
| Overfitting riski | Ağaç sayısı arttıkça artmaz | Ağaç sayısı çok artarsa artar → **erken durdurma** |
| Birleştirme | Ortalama / çoğunluk oyu | Ağırlıklı toplam: $F_m = F_{m-1} + \eta\, h_m$ |

## B.1) Bagging: varyansı ortalamayla azaltmak

Tek bir derin karar ağacı veriye çok duyarlıdır: verinin biraz farklı bir örneklemi bambaşka bir ağaç üretir. Bagging bu ağaçların ortalamasını alarak tahmini yumuşatır.

In [ ]:
rng = np.random.RandomState(0)
x_1b = np.sort(rng.rand(120) * 6)
y_1b = np.sin(x_1b) + rng.normal(0, 0.35, len(x_1b))
x_izgara = np.linspace(0, 6, 500)[:, None]

fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)

for i in range(6):
    idx = rng.randint(0, len(x_1b), len(x_1b))                    # bootstrap örneklemi
    agac = DecisionTreeRegressor().fit(x_1b[idx, None], y_1b[idx])
    eksenler[0].plot(x_izgara, agac.predict(x_izgara), alpha=0.6, lw=1)
eksenler[0].scatter(x_1b, y_1b, s=10, color="black")
eksenler[0].set_title("6 farklı bootstrap örnekleminde eğitilmiş tek ağaçlar\n(her biri çok farklı → yüksek varyans)")

rf = RandomForestRegressor(n_estimators=300, random_state=0).fit(x_1b[:, None], y_1b)
eksenler[1].scatter(x_1b, y_1b, s=10, color="black")
eksenler[1].plot(x_izgara, np.sin(x_izgara), "g--", label="gerçek fonksiyon")
eksenler[1].plot(x_izgara, rf.predict(x_izgara), "r", lw=2, label="Random Forest (300 ağaç ortalaması)")
eksenler[1].set_title("Bagging: ortalama alınca tahmin yumuşar")
eksenler[1].legend()
plt.tight_layout(); plt.show()

## B.2) Gradient Boosting'i sıfırdan yazmak

Kare hata kaybında $L = \frac{1}{2}(y - F(x))^2$ negatif gradyan tam olarak **artıktır**: $-\frac{\partial L}{\partial F} = y - F(x)$.

Algoritma:
1. $F_0(x) = \bar y$ (ortalamayla başla)
2. $m = 1 \dots M$ için:
   - Artıkları hesapla: $r_i = y_i - F_{m-1}(x_i)$
   - Sığ bir ağacı $r$'ye uydur: $h_m$
   - Güncelle: $F_m = F_{m-1} + \eta \cdot h_m$ ($\eta$ = öğrenme oranı)

In [ ]:
class BasitGradientBoosting:
    def __init__(self, n_agac=100, ogrenme_orani=0.1, derinlik=2):
        self.n_agac = n_agac
        self.ogrenme_orani = ogrenme_orani
        self.derinlik = derinlik

    def fit(self, X, y):
        self.baslangic = y.mean()
        self.agaclar = []
        F = np.full(len(y), self.baslangic)
        for _ in range(self.n_agac):
            artik = y - F                                   # negatif gradyan
            agac = DecisionTreeRegressor(max_depth=self.derinlik).fit(X, artik)
            F += self.ogrenme_orani * agac.predict(X)
            self.agaclar.append(agac)
        return self

    def predict(self, X, n_agac=None):
        F = np.full(len(X), self.baslangic)
        for agac in self.agaclar[:n_agac]:
            F += self.ogrenme_orani * agac.predict(X)
        return F

gb = BasitGradientBoosting(n_agac=200, ogrenme_orani=0.1, derinlik=2).fit(x_1b[:, None], y_1b)

fig, eksenler = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for ax, m in zip(eksenler, [1, 5, 30, 200]):
    ax.scatter(x_1b, y_1b, s=8, color="black", alpha=0.6)
    ax.plot(x_izgara, np.sin(x_izgara), "g--")
    ax.plot(x_izgara, gb.predict(x_izgara, n_agac=m), "r", lw=2)
    ax.set_title(f"{m} ağaçtan sonra")
fig.suptitle("Her yeni sığ ağaç, öncekilerin hatasını biraz daha düzeltiyor", y=1.03)
plt.tight_layout(); plt.show()

### XGBoost ve LightGBM ne ekliyor?

| | **XGBoost** | **LightGBM** |
|---|---|---|
| Gradyan bilgisi | 1. **ve 2. türev** (Hessian) — Newton adımı | 1. ve 2. türev |
| Regularizasyon | Yaprak ağırlıklarına L1 (`alpha`) ve L2 (`lambda`), bölme cezası `gamma` | `lambda_l1`, `lambda_l2`, `min_gain_to_split` |
| Ağaç büyütme | Seviye seviye (**level-wise**) | Yaprak yaprak (**leaf-wise**) — kaybı en çok düşüren yaprağı böler |
| Hız hilesi | `tree_method="hist"`, eksik değer için öğrenilen varsayılan yön | Histogram, **GOSS** (küçük gradyanlı örnekleri seyrekleştirme), **EFB** (seyrek özellikleri paketleme) |
| Kategorik özellik | `enable_categorical=True` | Yerel destek |

## B.3) Karşılaştırma: Karar Ağacı vs Random Forest vs XGBoost vs LightGBM

In [ ]:
X_b, y_b = make_classification(n_samples=40_000, n_features=40, n_informative=15, n_redundant=10,
                               flip_y=0.03, class_sep=0.8, random_state=7)
Xb_egt, Xb_tst, yb_egt, yb_tst = train_test_split(X_b, y_b, test_size=0.25, random_state=7)

modeller = {
    "Karar Ağacı": DecisionTreeClassifier(random_state=0),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0),
    "XGBoost": xgb.XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=6,
                                 subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                                 eval_metric="logloss", random_state=0, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(n_estimators=400, learning_rate=0.1, num_leaves=63,
                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                   random_state=0, n_jobs=-1, verbose=-1),
}

sonuclar = []
for isim, model in modeller.items():
    t0 = time.time()
    model.fit(Xb_egt, yb_egt)
    sure = time.time() - t0
    olasilik = model.predict_proba(Xb_tst)[:, 1]
    tahmin = (olasilik >= 0.5).astype(int)
    sonuclar.append({"Model": isim,
                     "Eğitim doğruluğu": accuracy_score(yb_egt, model.predict(Xb_egt)),
                     "Test doğruluğu": accuracy_score(yb_tst, tahmin),
                     "Test F1": f1_score(yb_tst, tahmin),
                     "Test ROC-AUC": roc_auc_score(yb_tst, olasilik),
                     "Eğitim süresi (sn)": sure})

sonuc_df = pd.DataFrame(sonuclar).set_index("Model").round(3)
sonuc_df

In [ ]:
fig, eksenler = plt.subplots(1, 2, figsize=(14, 4))
sonuc_df[["Eğitim doğruluğu", "Test doğruluğu"]].plot.bar(ax=eksenler[0], rot=0, ylim=(0.7, 1.01))
eksenler[0].set_title("Eğitim vs test doğruluğu\n(tek ağacın %100 eğitim skoru = overfitting)")
sonuc_df["Eğitim süresi (sn)"].plot.bar(ax=eksenler[1], rot=0, color="tab:orange")
eksenler[1].set_title("Eğitim süresi")
plt.tight_layout(); plt.show()

## B.4) Boosting'de overfitting ve erken durdurma (early stopping)

Boosting'de ağaç sayısı arttıkça doğrulama hatası bir noktadan sonra **yükselmeye** başlar. Doğrulama setini izleyip iyileşme durunca eğitimi kesmek gerekir.

In [ ]:
Xb_e2, Xb_d2, yb_e2, yb_d2 = train_test_split(Xb_egt, yb_egt, test_size=0.2, random_state=1)

fig, eksenler = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
for ax, lr_degeri in zip(eksenler, [0.3, 0.03]):
    model = xgb.XGBClassifier(n_estimators=1500, learning_rate=lr_degeri, max_depth=8,
                              tree_method="hist", eval_metric="logloss",
                              early_stopping_rounds=100, random_state=0, n_jobs=-1)
    model.fit(Xb_e2, yb_e2, eval_set=[(Xb_e2, yb_e2), (Xb_d2, yb_d2)], verbose=False)
    gecmis = model.evals_result()
    ax.plot(gecmis["validation_0"]["logloss"], label="eğitim")
    ax.plot(gecmis["validation_1"]["logloss"], label="doğrulama")
    ax.axvline(model.best_iteration, color="red", ls="--", label=f"en iyi iterasyon = {model.best_iteration}")
    ax.set_title(f"XGBoost, learning_rate = {lr_degeri}")
    ax.set_xlabel("ağaç sayısı"); ax.set_ylabel("log loss")
    ax.legend()
plt.tight_layout(); plt.show()

LightGBM'de aynı şey `callbacks` ile yapılır:

In [ ]:
lgb_model = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.05, num_leaves=63,
                               random_state=0, verbose=-1)
lgb_model.fit(Xb_e2, yb_e2, eval_set=[(Xb_d2, yb_d2)], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])

print("LightGBM en iyi iterasyon:", lgb_model.best_iteration_)
print(f"Test doğruluğu: {accuracy_score(yb_tst, lgb_model.predict(Xb_tst)):.3f}")

## B.5) Özellik önemleri

In [ ]:
onemler = pd.DataFrame({
    "Random Forest": modeller["Random Forest"].feature_importances_,
    "XGBoost": modeller["XGBoost"].feature_importances_,
    "LightGBM": modeller["LightGBM"].feature_importances_ / modeller["LightGBM"].feature_importances_.sum(),
}, index=[f"x{i}" for i in range(X_b.shape[1])])

en_onemli = onemler.mean(axis=1).sort_values(ascending=False).index[:12]
onemler.loc[en_onemli].plot.barh(figsize=(10, 6)).invert_yaxis()
plt.title("En önemli 12 özellik (normalize edilmiş önem)")
plt.xlabel("önem"); plt.show()

## B.6) Dengesiz veriye geri dönüş: boosting + `scale_pos_weight`

XGBoost'ta `scale_pos_weight = negatif / pozitif` sayısı, `class_weight="balanced"`'ın karşılığıdır.

In [ ]:
oran = (y_egt == 0).sum() / (y_egt == 1).sum()
print(f"scale_pos_weight = {oran:.1f}")

xgb_dengesiz = xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=5,
                                 scale_pos_weight=oran, eval_metric="aucpr",
                                 tree_method="hist", random_state=0, n_jobs=-1).fit(X_egt, y_egt)
lgb_dengesiz = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31,
                                  is_unbalance=True, random_state=0, verbose=-1).fit(X_egt, y_egt)

tablo_2 = pd.DataFrame([
    metrikler("LogReg balanced", y_tst, lr_dengeli.predict_proba(X_tst)[:, 1]),
    metrikler("XGBoost + scale_pos_weight", y_tst, xgb_dengesiz.predict_proba(X_tst)[:, 1]),
    metrikler("LightGBM + is_unbalance", y_tst, lgb_dengesiz.predict_proba(X_tst)[:, 1]),
]).round(3)
tablo_2

---
## Özet

- Dengesiz veride **accuracy'ye bakma**; Precision, Recall, F1 ve PR-AUC raporla.
- Hangi hatanın daha pahalı olduğuna göre (FP mi, FN mi?) metriği ve **eşiği** seç; eşiği doğrulama setinde ayarla.
- `class_weight="balanced"`, `scale_pos_weight`, `is_unbalance` azınlık sınıfını kayıpta ağırlıklandırır.
- **Bagging (Random Forest)**: derin ağaçların ortalaması → varyans düşer, ayarı kolay, paralel.
- **Boosting (XGBoost / LightGBM)**: sığ ağaçlar sırayla artıkları düzeltir → tablo verisinde genelde en güçlü yöntem; `learning_rate` + **early stopping** ile overfitting kontrol edilir.